# FRASCI Main: four-fragment LASSCF → TrimCI → LASSI/LASSIS

This is the reproducible H1-diagonal four-fragment workflow for
`h1diag_rev_block_6x8x10x12`.

It contains:

1. CAS-LASSCF/CSF control
2. TrimCI-LASSCF cold start
3. TrimCI-LASSCF warm start from the CAS orbitals
4. LASSI and LASSIS from CAS, cold, and warm checkpoints
5. LASSIS charge/spin expansions for `nspin = 0, 1, 2`

The notebook defaults to **reuse mode**, loading the checked-in June 16, 2026
results without launching the multi-hour calculations. Set `RUN_EXPENSIVE = True`
only when a fresh calculation is intentional.

In [ ]:
from pathlib import Path
from datetime import datetime
import json
import os
import subprocess
import time

from IPython.display import Markdown, display

ROOT = Path.cwd().resolve()
if ROOT.name != "FRASCI":
    candidate = next((p for p in [ROOT, *ROOT.parents] if (p / "FRASCI").is_dir()), None)
    if candidate is None:
        raise RuntimeError("Open this notebook from the FRASCI repository.")
    ROOT = candidate / "FRASCI"

PYTHON = ROOT.parent / "FRASCIenv" / "bin" / "python"
if not PYTHON.exists():
    PYTHON = Path(os.sys.executable)

FCIDUMP = ROOT / "data" / "fcidump_cycle_6"
FRAGMENT_JSON = ROOT / "data" / "partitions" / "h1diag_rev_block_6x8x10x12.json"
PARTITION = "h1diag_rev_block_6x8x10x12"

EXISTING_RUN = ROOT / "Outputs" / "lasscf" / "h1_4frag_pipeline_20260616_122809"
RUN_EXPENSIVE = False
FORCE_RERUN = False
RUN_ROOT = (
    ROOT / "Outputs" / "lasscf" / f"h1_4frag_pipeline_{datetime.now():%Y%m%d_%H%M%S}"
    if RUN_EXPENSIVE
    else EXISTING_RUN
)

CAS_MAX_CYCLE = 100
TRIMCI_MAX_CYCLE_MACRO = 100
TRIMCI_THRESHOLD = 0.01
TRIMCI_MAX_DETS = 2000
TRIMCI_MAX_ROUNDS = 4
LASSI_OPT = 1
LASSIS_NCHARGE = "s"
LASSIS_NSPINS = (0, 1, 2)

assert FCIDUMP.exists(), FCIDUMP
assert FRAGMENT_JSON.exists(), FRAGMENT_JSON
if not RUN_EXPENSIVE:
    assert RUN_ROOT.exists(), RUN_ROOT

print("Mode:", "RUN fresh calculations" if RUN_EXPENSIVE else "REUSE existing outputs")
print("Run root:", RUN_ROOT)
print("Python:", PYTHON)

## Helpers

In [ ]:
def load_json(path):
    path = Path(path)
    if not path.exists():
        return None
    with path.open() as handle:
        return json.load(handle)


def run_cmd(args, out_dir, done_file):
    out_dir = Path(out_dir)
    done_file = Path(done_file)
    out_dir.mkdir(parents=True, exist_ok=True)
    if done_file.exists() and not FORCE_RERUN:
        print("[reuse]", done_file)
        return 0
    if not RUN_EXPENSIVE:
        raise RuntimeError(f"Missing reusable result: {done_file}")

    log_path, err_path = out_dir / "run.log", out_dir / "run.err"
    env = os.environ.copy()
    env["PYTHONPATH"] = str(ROOT) + os.pathsep + env.get("PYTHONPATH", "")
    print("[run]", " ".join(map(str, args)))
    started = time.time()
    with log_path.open("w") as stdout, err_path.open("w") as stderr:
        result = subprocess.run(
            [str(x) for x in args],
            cwd=ROOT,
            env=env,
            stdout=stdout,
            stderr=stderr,
            text=True,
        )
    print(f"[done] rc={result.returncode}; wall={(time.time() - started) / 3600:.2f} h")
    if not done_file.exists():
        raise RuntimeError(f"Expected output was not written: {done_file}. See {err_path}")
    return result.returncode


def show(label, result):
    if result is None:
        display(Markdown(f"**{label}: missing**"))
        return
    wanted = (
        "status", "converged", "e_tot", "e_lasscf", "e_lasci",
        "e_lassi", "e_lassis", "delta_e_lassis",
        "wall_clock_sec", "wall_time_total",
    )
    lines = [f"### {label}"] + [
        f"- `{key}`: `{result[key]}`" for key in wanted if key in result
    ]
    display(Markdown("\n".join(lines)))

## Output layout

In [ ]:
cas_dir = RUN_ROOT / "cas_csf_control"
cold_dir = RUN_ROOT / "trimci_cold_thr0.01_dets2000_rounds4"
warm_dir = RUN_ROOT / "trimci_warm_from_cas_thr0.01_dets2000_rounds4"

sources = {"cas": cas_dir, "cold": cold_dir, "warm": warm_dir}
for label, path in sources.items():
    print(f"{label:>4}:", path)

## 1. CAS-LASSCF/CSF control

In [ ]:
run_cmd(
    [
        PYTHON, "-m", "FRASCI.lasscf.runners.run_lasscf_csf",
        "--fcidump", FCIDUMP,
        "--partition", PARTITION,
        "--fragment-orbs-json", FRAGMENT_JSON,
        "--max-cycle", CAS_MAX_CYCLE,
        "--output-dir", cas_dir,
    ],
    cas_dir,
    cas_dir / "result.json",
)
cas_result = load_json(cas_dir / "result.json")
show("CAS-LASSCF/CSF", cas_result)

## 2. TrimCI-LASSCF cold start

In [ ]:
run_cmd(
    [
        PYTHON, "-m", "FRASCI.lasscf.runners.run_lasscf_trimci",
        "--fcidump", FCIDUMP,
        "--partition", PARTITION,
        "--fragment-orbs-json", FRAGMENT_JSON,
        "--trimci-threshold", TRIMCI_THRESHOLD,
        "--trimci-max-dets", TRIMCI_MAX_DETS,
        "--trimci-max-rounds", TRIMCI_MAX_ROUNDS,
        "--max-cycle-macro", TRIMCI_MAX_CYCLE_MACRO,
        "--output-dir", cold_dir,
    ],
    cold_dir,
    cold_dir / "result.json",
)
cold_result = load_json(cold_dir / "result.json")
show("TrimCI-LASSCF cold", cold_result)

## 3. TrimCI-LASSCF warm start from CAS

In [ ]:
run_cmd(
    [
        PYTHON, "-m", "FRASCI.lasscf.runners.run_lasscf_trimci",
        "--fcidump", FCIDUMP,
        "--partition", PARTITION,
        "--fragment-orbs-json", FRAGMENT_JSON,
        "--init-from", cas_dir,
        "--trimci-threshold", TRIMCI_THRESHOLD,
        "--trimci-max-dets", TRIMCI_MAX_DETS,
        "--trimci-max-rounds", TRIMCI_MAX_ROUNDS,
        "--max-cycle-macro", TRIMCI_MAX_CYCLE_MACRO,
        "--output-dir", warm_dir,
    ],
    warm_dir,
    warm_dir / "result.json",
)
warm_result = load_json(warm_dir / "result.json")
show("TrimCI-LASSCF warm from CAS", warm_result)

## 4. LASSI/LASSIS from all starts and spin expansions

In [ ]:
interfragment = {}
for source, checkpoint_dir in sources.items():
    for nspin in LASSIS_NSPINS:
        out_dir = RUN_ROOT / f"lassi_lassis_from_{source}_ncharges_nspin{nspin}"
        args = [
            PYTHON, "-m", "FRASCI.lasscf.runners.run_lassi_lassis",
            "--lasscf-checkpoint-dir", checkpoint_dir,
            "--fcidump", FCIDUMP,
            "--output-dir", out_dir,
            "--opt", LASSI_OPT,
            "--lassis-ncharge", LASSIS_NCHARGE,
            "--lassis-nspin", nspin,
        ]
        # LASSI uses the same CT rootspace construction for every nspin.
        # Run it once (nspin=0); the nspin=1,2 branches add only LASSIS spaces.
        if nspin > 0:
            args.append("--skip-lassi")
        run_cmd(args, out_dir, out_dir / "summary.json")
        interfragment[(source, nspin)] = load_json(out_dir / "summary.json")

print(f"Loaded {sum(v is not None for v in interfragment.values())}/9 summaries")

## 5. Compact verification table

In [ ]:
lines = [
    "| Start | nspin | LASCI (Ha) | LASSI (Ha) | LASSIS (Ha) | LASSIS lowering (mHa) |",
    "|---|---:|---:|---:|---:|---:|",
]
for (source, nspin), result in interfragment.items():
    def fmt(key):
        value = result.get(key) if result else None
        return "—" if value is None else f"{value:.9f}"
    lowering = None if not result else result.get("delta_e_lassis")
    lowering_text = "—" if lowering is None else f"{1000 * lowering:.3f}"
    lines.append(
        f"| {source} | {nspin} | {fmt('e_lasci')} | {fmt('e_lassi')} | "
        f"{fmt('e_lassis')} | {lowering_text} |"
    )
display(Markdown("\n".join(lines)))

## Interpretation guardrails

- The three LASSCF calculations reached the configured 100-cycle limit and are
  recorded as `NOT_CONVERGED`; their energies are useful checkpoint comparisons,
  not fully converged variational endpoints.
- LASSI is attempted only for `nspin=0`. Its saved result includes an error record
  if the state-average construction fails; missing energy is not plotted as zero.
- `FRASCI_Results.ipynb` is the presentation notebook. It reads these files and
  avoids expensive solver calls.